# Dynamic Term Structure

**Purpose:** Introduce the dynamic term-structure helpers in `finstack.core.market_data.dtsm` using a synthetic yield panel.

**Prerequisites:** Familiarity with yield curves, Nelson-Siegel factors, and the intuition behind PCA on curve changes.

**In this notebook:** We simulate a panel of curves, extract Diebold-Li factors, build a forecast, run PCA, and generate a simple shocked-curve scenario.


## Concept

This notebook keeps the term-structure workflow compact but complete:

1. Simulate a realistic panel of curves.
2. Recover the level, slope, and curvature factors.
3. Forecast the next curve from the factor dynamics.
4. Compare that factor view to a PCA shock view on yield changes.


In [ ]:
import numpy as np

from finstack.core.market_data import dtsm


def ns_curve(beta1: float, beta2: float, beta3: float, lam: float, tenors: np.ndarray) -> np.ndarray:
    lt = lam * tenors
    slope_load = np.where(np.abs(lt) < 1e-10, 1.0, (1.0 - np.exp(-lt)) / lt)
    curv_load = slope_load - np.exp(-lt)
    return beta1 + beta2 * slope_load + beta3 * curv_load


def simulate_panel(n_dates: int, tenors: np.ndarray, lam: float, rng: np.random.Generator) -> np.ndarray:
    mu = np.array([0.040, -0.015, 0.005])
    phi = np.array([0.98, 0.95, 0.92])
    sigma = np.array([0.0015, 0.0020, 0.0025])
    betas = np.empty((n_dates, 3))
    betas[0] = mu
    for idx in range(1, n_dates):
        betas[idx] = mu + phi * (betas[idx - 1] - mu) + rng.normal(size=3) * sigma
    return np.vstack([ns_curve(beta[0], beta[1], beta[2], lam, tenors) for beta in betas])


## Factors, forecast, and scenario shock

The next cell works through the full script in one pass. It first builds a synthetic panel, then extracts factors and forecasts, and finally compares the factor approach with a PCA-based shock to the last observed curve.


In [ ]:
rng = np.random.default_rng(seed=20260416)
tenors = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
lam = 0.0609
yields = simulate_panel(200, tenors, lam, rng)

print(f"Yield panel shape  : {yields.shape}")
print(f"Mean 10Y yield     : {yields[:, 7].mean():.4%}")

factors = dtsm.diebold_li_fit_factors(tenors.tolist(), yields.tolist(), lam)
print("\nDiebold-Li factor summary")
print(f"beta1 mean/std     : {np.mean(factors['beta1']):+.4%} / {np.std(factors['beta1']):.4%}")
print(f"beta2 mean/std     : {np.mean(factors['beta2']):+.4%} / {np.std(factors['beta2']):.4%}")
print(f"beta3 mean/std     : {np.mean(factors['beta3']):+.4%} / {np.std(factors['beta3']):.4%}")
print(f"avg R^2            : {factors['r_squared_avg']:.6f}")

forecast = dtsm.diebold_li_forecast(tenors.tolist(), yields.tolist(), 12, lam)
print("\n12-step forecast")
for tenor, last, predicted in zip(tenors, yields[-1], forecast['forecast_yields']):
    print(f"  {tenor:>5.2f}y last={last:.4%} forecast={predicted:.4%}")

changes = np.diff(yields, axis=0)
pca = dtsm.yield_pca_fit(changes.tolist(), 3)
print("\nPCA explained variance")
for idx, share in enumerate(pca['explained_variance_ratio'], start=1):
    print(f"  PC{idx}: {share:.4f}")
print(f"Cumulative through PC3: {pca['cumulative_variance'][-1]:.4f}")

shock = np.asarray(dtsm.yield_pca_scenario(changes.tolist(), 0, 2.0, 3))
shocked_curve = yields[-1] + shock
print("\nPC1 +2 sigma shock (bp)")
for tenor, delta in zip(tenors, shock):
    print(f"  {tenor:>5.2f}y delta={delta * 1e4:+.2f} bp")


## Takeaways

- The `dtsm` module supports both a factor-model view and a PCA view of the curve.
- Diebold-Li gives interpretable level, slope, and curvature factors, while PCA gives empirical shock directions.
- Forecasting and scenario generation are close neighbors in practice, so it is useful that both workflows live in the same API area.


In [ ]:
{
    "avg_r_squared": round(factors['r_squared_avg'], 6),
    "pc3_cumulative": round(pca['cumulative_variance'][-1], 6),
    "final_10y": round(float(yields[-1][7]), 6),
    "shocked_10y": round(float(shocked_curve[7]), 6),
}
